# Agent A: Data Ingestion & Feature Engineering
## Kaggle Notebook - Data Preparation

This notebook prepares forex data for model training in the Kaggle environment.

In [ ]:
# Check Kaggle environment and GPU
import os
import sys

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') is not None
print(f"Kaggle Environment: {IS_KAGGLE}")

# Install dependencies
!pip install -q polars pyarrow joblib optuna ta-lib-bin

# GPU check
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Kaggle datasets
import subprocess

datasets = [
    "yakashri/export-15-fx-currency-pairs-1m",
    "imetomi/eur-usd-forex-pair-historical-data"
]

for dataset in datasets:
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', '-d', dataset, '-p', f'/kaggle/input/{dataset.split("/")[-1]}', '--unzip'],
        capture_output=True, text=True
    )
    print(f"Dataset {dataset}: {'OK' if result.returncode == 0 else 'FAILED'}")

In [ ]:
# Setup paths
import sys
sys.path.insert(0, '/kaggle/working')

# Clone repository if needed
if not os.path.exists('/kaggle/working/src'):
    !git clone https://github.com/user/forex-ml-bot.git /kaggle/working/

from src.agents.agent_a_data import DataIngestionEngine

# Initialize agent with Kaggle mode
agent = DataIngestionEngine(
    config_path='config/system_config.yaml',
    kaggle_mode=True
)

# Execute data pipeline
result = agent.execute()
print(f"\nAgent A Result: {result}")

In [ ]:
# Verify output
import pandas as pd

output_dir = '/kaggle/working/output/agent_a/features'
if os.path.exists(output_dir):
    files = os.listdir(output_dir)
    print(f"Feature files: {len(files)}")
    for f in files[:5]:
        print(f"  {f}")
    
    # Show sample
    if files:
        sample = pd.read_parquet(os.path.join(output_dir, files[0]))
        print(f"\nSample shape: {sample.shape}")
        print(f"Columns: {list(sample.columns[:10])}...")

In [ ]:
# Commit output to Kaggle dataset
import json

# Create metadata for new dataset
metadata = {
    "title": "forex-features-engineered",
    "id": "user/forex-features-engineered",
    "licenses": [{"name": "CC0-1.0"}]
}

with open('/kaggle/working/dataset-metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("Dataset metadata created. Use Kaggle API to publish.")